# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Areebaarain/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Freshness Multiplier

**Finding:** FlyRank reports that freshness is associated with better search performance. In particular, the report found a significant difference in impressions between refreshed and stale pages (p < 0.001).

**Where does the label come from?**
The refreshed/stale label comes from FlyRank's internal update and freshness logic. An update can include changes to titles, descriptions, body content, internal links, images, widgets, or layout.

**Does the validation design carry the claim?**
The validation supports the claim that refreshed and stale pages have different impression levels. However, the design is observational, so I would avoid saying that refreshing a page alone caused the improvement. Other factors, such as content age, topic demand, or page quality, could also contribute. A controlled before/after or experimental design would make the causal claim stronger.

### Finding 2 — The CTR Cliff

**Finding:** FlyRank found that CTR differs strongly by search-position tier. Weighted CTR was 0.420% for Top 3 pages, 0.340% for Page 1, 0.163% for positions 21–50, and 0.050% for Deep pages. The position-tier comparison was statistically significant (p < 0.001).

**Where does the label come from?**
The position-tier label comes from observed average search position. CTR is calculated from observed clicks and impressions.

**Does the validation design carry the claim?**
The validation strongly supports the observed relationship between search position and CTR. However, it is still observational, so it does not prove that moving a page to a higher position will automatically cause a specific CTR increase. Other factors, such as title and meta-description quality, can also affect clicks.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
!git clone https://github.com/Areebaarain/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [7]:

import pandas as pd
import os

path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(path))

df = pd.read_csv(path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

File exists: True
Rows: 30000
Columns: 44


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================
# ML-09 Section 2
# Honest client-grouped split
# Same Decision Tree as Week-5
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# --------------------------------------------
# 1. Copy data
# --------------------------------------------

model_df = df.copy()

# --------------------------------------------
# 2. Same target and features as Week-5
# --------------------------------------------

target = "trend_direction"

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

# Keep client_id for grouped split
model_df = model_df[
    features + [target, "client_id"]
].dropna()

# --------------------------------------------
# 3. Encode target
# --------------------------------------------

label_encoder = LabelEncoder()
model_df[target] = label_encoder.fit_transform(model_df[target])

X = model_df[features]
y = model_df[target]
groups = model_df["client_id"]

# --------------------------------------------
# 4. Honest grouped split by CLIENT
# --------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())

# Check that no client appears in both sets
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Overlapping clients:", len(train_clients & test_clients))

# --------------------------------------------
# 5. Baseline — majority class
# --------------------------------------------

majority_class = y_train.mode()[0]

baseline_predictions = np.full(
    len(y_test),
    majority_class
)

# --------------------------------------------
# 6. SAME Week-5 Decision Tree
# --------------------------------------------

tree_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)

# --------------------------------------------
# 7. Compare both models
# --------------------------------------------

results_honest = pd.DataFrame({

    "Model": [
        "Week-4 Baseline (Majority Class)",
        "Decision Tree (Depth 4)"
    ],

    "Accuracy": [
        accuracy_score(y_test, baseline_predictions),
        accuracy_score(y_test, tree_predictions)
    ],

    "Precision (weighted)": [
        precision_score(
            y_test,
            baseline_predictions,
            average="weighted",
            zero_division=0
        ),
        precision_score(
            y_test,
            tree_predictions,
            average="weighted",
            zero_division=0
        )
    ],

    "Recall (weighted)": [
        recall_score(
            y_test,
            baseline_predictions,
            average="weighted",
            zero_division=0
        ),
        recall_score(
            y_test,
            tree_predictions,
            average="weighted",
            zero_division=0
        )
    ],

    "F1 Score (weighted)": [
        f1_score(
            y_test,
            baseline_predictions,
            average="weighted",
            zero_division=0
        ),
        f1_score(
            y_test,
            tree_predictions,
            average="weighted",
            zero_division=0
        )
    ]
})

results_honest = results_honest.round(3)

print("\nHONEST CLIENT-GROUPED RESULTS:")
display(results_honest)

Training rows: 14234
Test rows: 5663
Training clients: 23
Test clients: 6
Overlapping clients: 0

HONEST CLIENT-GROUPED RESULTS:


,Model,Accuracy,Precision (weighted),Recall (weighted),F1 Score (weighted)
0,Week-4 Baseline (Majority Class),0.624,0.389,0.624,0.479
1,Decision Tree (Depth 4),0.625,0.477,0.625,0.502


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================
# 3. Leakage audit
# Final feature set vs target
# ============================================

target = "trend_direction"

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate"
]

# --------------------------------------------
# 1. Direct target leakage check
# --------------------------------------------

leakage_features = [
    col for col in features
    if col == target or col in ["trend_direction", "trend_pct"]
]

print("Target:", target)
print("Number of features:", len(features))
print("Potential direct leakage columns:", leakage_features)

# --------------------------------------------
# 2. Check for target-related columns
# --------------------------------------------

suspicious_columns = [
    col for col in features
    if any(word in col.lower()
           for word in ["trend", "label", "target"])
]

print("\nSuspicious feature names:")
print(suspicious_columns)

# --------------------------------------------
# 3. Final verdict
# --------------------------------------------

if len(leakage_features) == 0:
    print("\nVERDICT: PASS — no direct target leakage found.")
else:
    print("\nVERDICT: FAIL — remove the leakage feature(s).")

Target: trend_direction
Number of features: 16
Potential direct leakage columns: []

Suspicious feature names:
[]

VERDICT: PASS — no direct target leakage found.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
### 4. Claim rewrite

**Original claim:**
The Decision Tree performed better than the baseline across all comparison metrics, showing that it learned useful patterns from the content and search-performance features.

**Safer rewrite:**
In the evaluated split, the Decision Tree showed higher accuracy and weighted F1 than the majority-class baseline. This is an observed result on the tested data and suggests that the selected features contain directional signal for identifying different trend categories. However, the result should be treated as decision-support evidence rather than proof that the model will perform equally well on unseen clients or future data.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.